# 세션3 — Post-retrieval / Rerank

세션1~2에서 Hybrid(BM25+Dense)와 Multi-Query, HyDE로 **검색기 자체와 질문을 개선**하는 방법을 봤습니다. 그런데 아무리 검색기를 잘 조합해도, 1차로 회수한 후보 문서들의 **순위**가 항상 정답 순이라는 보장은 없습니다.

이번 세션에서는 "1차로 찾아온 후보들을 어떻게 다시 정리할까?"를 세 단계로 봅니다.

1. **Threshold 필터링** — 점수가 낮은 문서를 그냥 잘라내면 될까?
2. **Bi-Encoder Reranker** — 후보를 독립적으로 다시 임베딩해서 정렬하면 나아질까?
3. **Cross-Encoder Reranker** — 질문과 문서를 "함께" 보고 관련성을 판단하면?

세 단계 모두 **같은 질문, 같은 후보 문서 묶음**으로 실제 순위가 어떻게 바뀌는지 직접 확인합니다. 세션1/2와 동일하게 키오스크 이용실태 조사 PDF와 `project/vectorstore`를 재사용합니다.

In [1]:
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# 세션1/2와 마찬가지로 project/ 폴더에 의존하지 않고 필요한 값을 이 노트북에서 직접 정의합니다.
DATA_PATH = os.path.join("..", "data", "키오스크(무인정보단말기) 이용실태 조사.pdf")
VECTORSTORE_DIR = os.path.join("project", "vectorstore")
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")
llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)

docs = PyPDFLoader(DATA_PATH).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
splited_docs = splitter.split_documents(docs)
print(f"원본 페이지 수: {len(docs)}, 청크 수: {len(splited_docs)}")


def show_results(docs, title):
    """검색 결과를 순위, 페이지 번호와 함께 짧게 출력합니다."""
    print(f"--- {title} ---")
    for i, doc in enumerate(docs):
        page = doc.metadata.get("page")
        preview = doc.page_content[:60].strip().replace(chr(10), " ")
        print(f"[{i + 1}순위][p{page}] {preview}...")
    print()


/var/folders/px/v7_qzrl907d919ql0sn3f74r0000gn/T/ipykernel_77574/1566478415.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


원본 페이지 수: 59, 청크 수: 73


## 세션1 복습 — Hybrid Retriever 다시 만들기 (후보를 넉넉히)

이 노트북도 별도 파일이라 세션1/2의 retriever가 메모리에 없습니다. 같은 설정으로 다시 만들되, 이번엔 **reranker가 재정렬할 후보를 더 넉넉히 받기 위해 k를 5로 늘립니다.** (세션1/2는 k=3)

이후 Threshold/Bi-Encoder/Cross-Encoder 세 단계 모두 **동일한 질문, 동일한 hybrid 후보 묶음**을 기준으로 비교합니다.

In [2]:
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

vectorstore = Chroma(persist_directory=VECTORSTORE_DIR, embedding_function=embeddings)

bm25_retriever = BM25Retriever.from_documents(splited_docs)
bm25_retriever.k = 5

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

query = "상담사례 접수 건수가 가장 많았던 키오스크 종류가 뭐야?"

candidates = hybrid_retriever.invoke(query)
show_results(candidates, f"Hybrid 후보 (rerank 전): '{query}'")


--- Hybrid 후보 (rerank 전): '상담사례 접수 건수가 가장 많았던 키오스크 종류가 뭐야?' ---
[1순위][p17] 키오스크(무인정보단말기)이용 실태조사 18 시장조사국 시장감시팀 Ⅳ소비자 불만·피해사례1. 현황□최근 4년...
[2순위][p2] 키오스크(무인정보단말기)이용 실태조사‘표’ 목차[표2-1-1] 키오스크의 종류 · · · · · · · ·...
[3순위][p18] 키오스크(무인정보단말기)이용 실태조사 19 시장조사국 시장감시팀 □(업종별 불만‧ 피해 유형 분석)업종별로...
[4순위][p2] · · · · · · · · · · · · · · · · 18[표4-2-2] 불만‧피해사례 유형(N=96)...
[5순위][p5] 키오스크(무인정보단말기)이용 실태조사 6 시장조사국 시장감시팀 □이에 정부는 ‘무인정보단말기 접근성 지침’을...
[6순위][p21] 키오스크(무인정보단말기)이용 실태조사 22 시장조사국 시장감시팀 Ⅴ소비자 설문조사▣ 설문대상: 키오스크 이용...
[7순위][p29] 키오스크(무인정보단말기)이용 실태조사 30 시장조사국 시장감시팀 ㅇ (외식업)소비자의 ‘주문 실수’로 인한...
[8순위][p3] · · · 52‘그림’ 목차[그림2-1-1] 키오스크 예시 사진 · · · · · · · · · · · ·...



19페이지([표4-2-4] 키오스크 종류별 접수 현황)에는 이런 내용이 있습니다.

> (종류) 소비자 불만·피해가 접수된 96건의 키오스크 종류를 분석한 결과 **‘무인주문기’가 34건(35.4%)**으로 가장 많았고, ...

즉 이 질문의 **정답은 19페이지(0-index로는 p18) 청크**입니다. 하지만 위 Hybrid 결과를 보면 정답 청크가 1위가 아니라 **3위 정도**로 밀려 있고, 1위 자리는 "소비자 불만·피해사례" 장의 도입부(18페이지, 정확한 종류별 수치는 없이 전체 96건이라는 개요만 있음)가 차지하고 있습니다. 그리고 표‧그림 목차 페이지(2~3페이지, 실제 내용은 없이 표 제목만 나열된 페이지)도 후보에 끼어 있습니다.

**"주제는 비슷하지만 실제 답은 없는 개요/목차 페이지"**가 **"실제 답이 있는 페이지"**보다 위에 있는 상황입니다. 이제 이 후보 묶음을 세 가지 방법으로 다시 정렬해보겠습니다.

## ① Threshold 필터링 — 점수로 자르면 될까?

가장 단순한 post-retrieval 방법은 "유사도 점수가 일정 기준(threshold) 이하인 문서는 버린다"입니다. `similarity_search_with_relevance_scores`로 Dense 점수를 직접 확인해봅니다.

In [3]:
scored_docs = vectorstore.similarity_search_with_relevance_scores(query, k=8)

print(f"--- Dense 유사도 점수: '{query}' ---")
for i, (doc, score) in enumerate(scored_docs):
    page = doc.metadata.get("page")
    preview = doc.page_content[:55].strip().replace(chr(10), " ")
    print(f"[{i + 1}순위][score={score:.3f}][p{page}] {preview}...")


--- Dense 유사도 점수: '상담사례 접수 건수가 가장 많았던 키오스크 종류가 뭐야?' ---
[1순위][score=0.439][p17] 키오스크(무인정보단말기)이용 실태조사 18 시장조사국 시장감시팀 Ⅳ소비자 불만·피해사례1. 현황□최...
[2순위][score=0.374][p2] · · · · · · · · · · · · · · · · 18[표4-2-2] 불만‧피해사례 유형(N...
[3순위][score=0.369][p2] 키오스크(무인정보단말기)이용 실태조사‘표’ 목차[표2-1-1] 키오스크의 종류 · · · · · ·...
[4순위][score=0.366][p21] 키오스크(무인정보단말기)이용 실태조사 22 시장조사국 시장감시팀 Ⅴ소비자 설문조사▣ 설문대상: 키오...
[5순위][score=0.348][p3] · · · 52‘그림’ 목차[그림2-1-1] 키오스크 예시 사진 · · · · · · · · · ·...
[6순위][score=0.344][p2] · · · · · · · · · · · · · · · · · 25[표5-2-1] 키오스크 이용 중...
[7순위][score=0.327][p30] 키오스크(무인정보단말기)이용 실태조사 31 시장조사국 시장감시팀 3. 키오스크 편의성‧접근성 개선...
[8순위][score=0.326][p29] 키오스크(무인정보단말기)이용 실태조사 30 시장조사국 시장감시팀 ㅇ (외식업)소비자의 ‘주문 실수’...


실제 점수를 보면 문제가 보입니다.

- 표‧그림 **목차 페이지(p2, p3)** — 실제 내용은 전혀 없이 표 제목만 나열된 페이지인데도 **0.35~0.37점**으로 꽤 높게 나옵니다. "키오스크의 종류", "불만·피해사례 유형" 같은 표 제목 단어들이 질문과 표면적으로 비슷하기 때문입니다.
- 반면 **진짜 정답이 있는 p18 청크는 이 상위 8개 안에 아예 들어오지 못했습니다.**

즉 threshold를 어디에 놓아도 딜레마가 생깁니다.
- 기준을 낮추면(예: 0.3) → 목차 페이지 같은 **가짜 관련 문서**까지 다 살아남습니다.
- 기준을 높이면(예: 0.4) → 목차 페이지는 걸러지지만, **애초에 정답 청크도 이 점수대 안에 없어서** threshold와 무관하게 못 찾습니다.

점수는 "질문과 문서가 표면적으로 얼마나 가까운가"만 볼 뿐, "이 문서가 질문에 실제로 답하는가"는 보지 못합니다. 그래서 점수 하나로 자르는 방식(threshold)만으로는 순위 문제를 해결할 수 없습니다.

## ② Bi-Encoder Reranker — 후보를 독립적으로 다시 임베딩하면?

Threshold보다 한 단계 나은 방법은, 1차로 회수한 후보들을 **다시 정렬**하는 것입니다. 가장 쉬운 재정렬 방법은 질문과 각 문서를 **각각 독립적으로 임베딩**한 뒤 코사인 유사도로 순위를 다시 매기는 것 — 이게 Bi-Encoder Reranker입니다. (Dense Retriever와 원리는 동일하고, "1차 후보 위에서 한 번 더" 적용한다는 점만 다릅니다.)

In [4]:
import numpy as np


def cosine_similarity(vec_a, vec_b):
    vec_a, vec_b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)))


def bi_encoder_rerank(query, docs):
    """질문과 문서를 각각 독립적으로 임베딩해 코사인 유사도로 재정렬합니다."""
    query_vec = embeddings.embed_query(query)
    scored = []
    for doc in docs:
        doc_vec = embeddings.embed_query(doc.page_content)
        scored.append((cosine_similarity(query_vec, doc_vec), doc))
    scored.sort(key=lambda x: -x[0])
    return scored


bi_encoder_reranked = bi_encoder_rerank(query, candidates)

print(f"--- Bi-Encoder Reranker 결과: '{query}' ---")
for i, (score, doc) in enumerate(bi_encoder_reranked):
    page = doc.metadata.get("page")
    preview = doc.page_content[:55].strip().replace(chr(10), " ")
    print(f"[{i + 1}순위][sim={score:.4f}][p{page}] {preview}...")


--- Bi-Encoder Reranker 결과: '상담사례 접수 건수가 가장 많았던 키오스크 종류가 뭐야?' ---
[1순위][sim=0.6032][p17] 키오스크(무인정보단말기)이용 실태조사 18 시장조사국 시장감시팀 Ⅳ소비자 불만·피해사례1. 현황□최...
[2순위][sim=0.5573][p2] · · · · · · · · · · · · · · · · 18[표4-2-2] 불만‧피해사례 유형(N...
[3순위][sim=0.5538][p2] 키오스크(무인정보단말기)이용 실태조사‘표’ 목차[표2-1-1] 키오스크의 종류 · · · · · ·...
[4순위][sim=0.5515][p21] 키오스크(무인정보단말기)이용 실태조사 22 시장조사국 시장감시팀 Ⅴ소비자 설문조사▣ 설문대상: 키오...
[5순위][sim=0.5390][p3] · · · 52‘그림’ 목차[그림2-1-1] 키오스크 예시 사진 · · · · · · · · · ·...
[6순위][sim=0.5235][p29] 키오스크(무인정보단말기)이용 실태조사 30 시장조사국 시장감시팀 ㅇ (외식업)소비자의 ‘주문 실수’...
[7순위][sim=0.5177][p18] 키오스크(무인정보단말기)이용 실태조사 19 시장조사국 시장감시팀 □(업종별 불만‧ 피해 유형 분석)...
[8순위][sim=0.4535][p5] 키오스크(무인정보단말기)이용 실태조사 6 시장조사국 시장감시팀 □이에 정부는 ‘무인정보단말기 접근성...


결과를 Hybrid(재정렬 전) 순위와 비교해보면, 정답 청크(p18)의 순위가 **나아지지 않고 오히려 더 뒤로 밀립니다.**

이유는 Bi-Encoder Reranker가 결국 Dense Retriever와 **같은 원리**이기 때문입니다. 질문과 문서를 따로따로 벡터로 만들고 거리만 재는 방식이라, "이 문서가 질문에 실제로 답하는 문장을 담고 있는가"까지는 판단하지 못합니다. Hybrid에서는 BM25가 힘을 보태 p18을 3위까지 끌어올렸지만, 순수 임베딩 유사도만으로 다시 정렬하니 그 이점이 사라진 것입니다.

## ③ Cross-Encoder Reranker — 질문과 문서를 "함께" 보면?

Bi-Encoder는 질문과 문서를 따로 인코딩하기 때문에 둘 사이의 **상호작용**을 볼 수 없습니다. Cross-Encoder Reranker는 **질문과 문서를 한 쌍으로 묶어서 동시에** 모델에 입력하고, "이 쌍이 얼마나 관련 있는지" 점수 하나를 직접 계산합니다. 후보 하나하나를 질문과 짝지어 정밀하게 채점하는 대신, 후보 수만큼 연산이 필요해 더 느립니다.

`langchain_classic`의 `CrossEncoderReranker`와 `BAAI/bge-reranker-v2-m3` 모델을 사용합니다.

In [5]:
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
top_n = 4
reranker = CrossEncoderReranker(model=cross_encoder, top_n=top_n)

cross_encoder_reranked = reranker.compress_documents(candidates, query)

show_results(candidates, f"rerank 이전 (Hybrid, {len(candidates)}개)")
show_results(cross_encoder_reranked, f"Cross-Encoder rerank 이후 (top {top_n}개)")


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

--- rerank 이전 (Hybrid, 8개) ---
[1순위][p17] 키오스크(무인정보단말기)이용 실태조사 18 시장조사국 시장감시팀 Ⅳ소비자 불만·피해사례1. 현황□최근 4년...
[2순위][p2] 키오스크(무인정보단말기)이용 실태조사‘표’ 목차[표2-1-1] 키오스크의 종류 · · · · · · · ·...
[3순위][p18] 키오스크(무인정보단말기)이용 실태조사 19 시장조사국 시장감시팀 □(업종별 불만‧ 피해 유형 분석)업종별로...
[4순위][p2] · · · · · · · · · · · · · · · · 18[표4-2-2] 불만‧피해사례 유형(N=96)...
[5순위][p5] 키오스크(무인정보단말기)이용 실태조사 6 시장조사국 시장감시팀 □이에 정부는 ‘무인정보단말기 접근성 지침’을...
[6순위][p21] 키오스크(무인정보단말기)이용 실태조사 22 시장조사국 시장감시팀 Ⅴ소비자 설문조사▣ 설문대상: 키오스크 이용...
[7순위][p29] 키오스크(무인정보단말기)이용 실태조사 30 시장조사국 시장감시팀 ㅇ (외식업)소비자의 ‘주문 실수’로 인한...
[8순위][p3] · · · 52‘그림’ 목차[그림2-1-1] 키오스크 예시 사진 · · · · · · · · · · · ·...

--- Cross-Encoder rerank 이후 (top 4개) ---
[1순위][p18] 키오스크(무인정보단말기)이용 실태조사 19 시장조사국 시장감시팀 □(업종별 불만‧ 피해 유형 분석)업종별로...
[2순위][p17] 키오스크(무인정보단말기)이용 실태조사 18 시장조사국 시장감시팀 Ⅳ소비자 불만·피해사례1. 현황□최근 4년...
[3순위][p2] 키오스크(무인정보단말기)이용 실태조사‘표’ 목차[표2-1-1] 키오스크의 종류 · · · · · · · ·...
[4순위][p29] 키오스크(무인정보단말기)이용 실태조사 30 시장조사국 시장감시팀 ㅇ (외식업)소비자의 ‘주문 실수’로 인한...



세 방식의 정답 청크(p18) 순위를 정리하면 이렇게 달라집니다.

| 방식 | 정답 청크(p18)의 순위 |
|---|---|
| Hybrid (재정렬 전) | 3위 |
| Bi-Encoder Reranker | 더 뒤로 밀림 (개선 없음) |
| **Cross-Encoder Reranker** | **1위로 승격** |

Cross-Encoder만 질문("접수 건수가 가장 많았던 **키오스크 종류**")과 각 후보 문서를 함께 보고, "종류별 수치가 실제로 담긴 문단"과 "종류를 언급만 하는 개요/목차 문단"을 구분해냅니다. 표면적인 단어·임베딩 유사도로는 구분되지 않던 차이를 잡아내는 것이 Cross-Encoder Reranker의 핵심입니다.

## 한계 — Reranker가 만능은 아닙니다

Cross-Encoder Reranker는 강력하지만 두 가지 한계가 있습니다.

1. **1차 후보에 없으면 애초에 살릴 수 없습니다.** Cross-Encoder는 `hybrid_retriever`가 회수한 후보 안에서만 순위를 다시 매깁니다. 만약 1차 검색(Hybrid) 단계에서 정답 청크가 후보에 아예 안 들어왔다면, 아무리 좋은 reranker를 써도 찾을 수 없습니다. 즉 **recall(놓치지 않고 찾아내는 정도)은 여전히 1차 retriever의 몫**이고, reranker는 그 안에서 **precision(순위)**만 개선합니다.
2. **후보 수만큼 비용이 듭니다.** 질문-문서 쌍마다 모델을 한 번씩 통과시켜야 하므로, 후보가 많아질수록 느려지고 비용도 커집니다.

`old/02_Rerank 기법` 노트북은 지금 만든 Cross-Encoder Reranker를 LangGraph 챗봇 그래프에 그대로 끼워 넣어, 1차 검색(top_n×5개) → rerank(top_n개) → 답변 생성까지 이어지는 흐름을 구현합니다. 여기서 확인한 "reranker는 후보 안에서만 재정렬한다"는 한계가, 다음 세션(Indexing)과 7교시(응답 품질 평가)에서 "애초에 검색이 실패하면 어떻게 감지하고 대응할지"를 다루는 이유로 자연스럽게 이어집니다.